# **Cell 1: Settings**

In [ ]:
# Everything you might change lives here.
BRANCH = "main"
SEEDS = [1, 2, 3, 4]     # seed 42 is the Step 5 model, already trained
EPOCHS = 8
OUT = "/kaggle/working/ensemble"

# For a short test: SMOKE = True. For the saved full run: SMOKE = False.
SMOKE = False

# **Cell 2: Clone the code**

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO = Path("/tmp/oct-calibration")
if not (REPO / "src" / "train.py").exists():
    subprocess.run(["rm", "-rf", str(REPO)])
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/zyanislam/oct-calibration.git", str(REPO)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f"git clone failed. Is Internet on in Settings?\n{r.stderr}"

# Python remembers "no module here" for a path added before the folder existed: clear it
sys.path_importer_cache.pop(str(REPO), None)
importlib.invalidate_caches()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("code at", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# **Cell 3: Copy the data to local disk**

In [ ]:
import shutil, time
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from tqdm.auto import tqdm

SOURCE = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256")
ROOT = Path("/tmp/oct256")   # local disk: fast, wiped when the session ends
assert SOURCE.is_dir(), f"dataset not found at {SOURCE}: attach it under Input"

paths = pd.read_csv(REPO / "splits/oct_split_fold0.csv")["path"]

def copy_one(rel):
    dst = ROOT / rel
    if not dst.exists():                       # rerunning skips files already copied
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(SOURCE / rel, dst)

t = time.time()
with ThreadPoolExecutor(max_workers=64) as pool:
    list(tqdm(pool.map(copy_one, paths), total=len(paths), desc="Copying"))
print(f"{len(paths)} files on local disk in {time.time() - t:.0f}s")

# **Cell 4: Train the models**

In [ ]:
import torch
from src.dataset import make_loaders
from src.model import build_model
from src.train import set_seed, class_weights, subset_loader, fit

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 x1 in Settings"
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

seeds = SEEDS[:1] if SMOKE else SEEDS
epochs = 1 if SMOKE else EPOCHS
start = time.time()

for seed in seeds:
    print(f"\n=== seed {seed} ===")
    set_seed(seed)     # before the loaders and the model, as in Step 5
    loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
    weights = class_weights(loaders["train"].dataset.labels)
    if SMOKE:
        loaders = dict(loaders, train=subset_loader(loaders["train"], 10_000, seed=seed))
    fit(build_model(), loaders, f"{OUT}/seed_{seed}", epochs=epochs, weights=weights, device=device)
    print(f"seed {seed} done, {(time.time() - start) / 60:.0f} min so far")

# **Cell 5: Check the results**

In [ ]:
from sklearn.metrics import f1_score

rows = []
for seed in seeds:
    log = pd.read_csv(f"{OUT}/seed_{seed}/log.csv")
    test = torch.load(f"{OUT}/seed_{seed}/logits_test.pt")
    rows.append({"seed": seed,
                 "best_val_macro_f1": log.val_macro_f1.max(),
                 "best_epoch": int(log.val_macro_f1.idxmax()) + 1,
                 "test_macro_f1": round(f1_score(test["labels"], test["logits"].argmax(1), average="macro"), 4)})
display(pd.DataFrame(rows))
print("Reference, seed 42: best val macro-F1 0.9388, test macro-F1 0.981")